<a href="https://colab.research.google.com/github/arsyad429/Indonesia-Glints-job-vacancy-Data-Cleaning-Using-PySpark/blob/main/Tugas_Akhir_Big_Data_Muhammad_Arsyad_Setiawan_245150207111052.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, isnan, when
import pandas as pd

In [ ]:
spark = SparkSession.builder.appName("tugas-akhir").getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_PATH = "/content/drive/MyDrive/Analitik_Big_Data"
DATA_PATH = f"{BASE_PATH}/MuhammadArsyadSetiawan_245150207111052_DataLowonganKerja_Updated.csv"


In [ ]:
df_pandas = pd.read_csv(DATA_PATH)
df_pandas.head()

,title,company,location,job_type,category,experience_level,education_level,salary_range,job_requirements,job_responsibilities,posted_date,url,source_platform
0,Maintenance Engineering (ME),Manna Kampus,Kab. Sleman,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > IT Support,Pengalaman kurang dari 1 tahun,Minimal Sarjana (S1),Perusahaan tidak menampilkan gaji,"Server Management, Networking, PostgreSQL, IT ...","🖥️ Tech Problem Solver Wanted!, Kami mencari t...",28-02-2026,https://glints.com/id/opportunities/jobs/maint...,glints.com
1,Full Stack Developer,PT. Upindo Mitra Sejati,Jakarta Barat,Paruh Waktu · Hybrid,Komputer & Perangkat Lunak > Full Stack Developer,1 - 3 tahun pengalaman,Minimal Diploma (D1 - D4),Rp1.000.000 - 3.000.000/Bulan,"MySQL, Node.js, React.js",Kandidat bertugas membuat aplikasi berbasis we...,28-01-2026,https://glints.com/id/opportunities/jobs/full-...,glints.com
2,Senior Fullstack Developer (Golang & React),PT Tirtamas Coldstorindo Logistik,Surabaya,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > Full Stack Developer,3 - 5 tahun pengalaman,Minimal SMA/SMK,Rp6.500.000 - 7.500.000/Bulan,"SDLC, React.js, Messege Brokers, PostgreSQL, F...",We are looking for a Senior Fullstack Develope...,28-02-2026,https://glints.com/id/opportunities/jobs/senio...,glints.com
3,Technical Support,PT Yogura Tekindo,Pekanbaru,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > Technical Support...,1 - 3 tahun pengalaman,Minimal Sarjana (S1),Perusahaan tidak menampilkan gaji,"Troubleshooting, Composite Materials, Technica...",Technical Support - PT. Indo Riau Perkasa (Yot...,28-02-2026,https://glints.com/id/opportunities/jobs/techn...,glints.com
4,Fullstack Developer (Intern),PT Tirtamas Coldstorindo Logistik,Surabaya,Penuh Waktu · Kerja di kantor,Komputer & Perangkat Lunak > Full Stack Developer,Pengalaman kurang dari 1 tahun,Minimal Diploma (D1 - D4),Perusahaan tidak menampilkan gaji,"Python, PostgreSQL, PHP, react js, MySQL, vue js",We are looking for a Fullstack Developer Inter...,28-02-2026,https://glints.com/id/opportunities/jobs/fulls...,glints.com


In [ ]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(DATA_PATH)
df.show(5)

+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+---------------+
|               title|             company|     location|            job_type|            category|    experience_level|     education_level|        salary_range|    job_requirements|job_responsibilities|posted_date|                 url|source_platform|
+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+---------------+
|Maintenance Engin...|        Manna Kampus|  Kab. Sleman|Penuh Waktu · Ker...|Komputer & Perang...|Pengalaman kurang...|Minimal Sarjana (S1)|Perusahaan tidak ...|Server Management...|🖥️ Tech Problem ...| 28-02-2026|https://glints.co...|  

# **EDA**

In [ ]:
df.dtypes

[('title', 'string'),
 ('company', 'string'),
 ('location', 'string'),
 ('job_type', 'string'),
 ('category', 'string'),
 ('experience_level', 'string'),
 ('education_level', 'string'),
 ('salary_range', 'string'),
 ('job_requirements', 'string'),
 ('job_responsibilities', 'string'),
 ('posted_date', 'string'),
 ('url', 'string'),
 ('source_platform', 'string')]

In [ ]:
print(f"Panjang baris: {df.count()}")
print(f"Panjang kolom: {len(df.columns)}")

Panjang baris: 16712
Panjang kolom: 13


In [ ]:
df.createOrReplaceTempView("job_vacancy")

In [ ]:
most_many_job_category = spark.sql("""
select category, COUNT(*) as category_count
from job_vacancy
GROUP BY category
ORDER BY category_count DESC
LIMIT 5
""")

most_many_job_category.show()

+--------------------+--------------+
|            category|category_count|
+--------------------+--------------+
|                NULL|          2305|
|Operasional & Pel...|          1130|
|Administrasi & HR...|           984|
|Desain > Desainer...|           709|
|Seni, Media, & Ko...|           672|
+--------------------+--------------+



In [ ]:
salary = spark.sql("""
select `salary_range` from job_vacancy
where `salary_range` is not null
LIMIT 20
""")

salary.show()

+--------------------+
|        salary_range|
+--------------------+
|Perusahaan tidak ...|
|Rp1.000.000 - 3.0...|
|Rp6.500.000 - 7.5...|
|Perusahaan tidak ...|
|Perusahaan tidak ...|
|Perusahaan tidak ...|
|Rp5.000.000 - 5.1...|
|   Rp1.000.000/Bulan|
| Kualifikasi Utam...|
|Rp5.729.876 - 6.0...|
|Rp12.000.000 - 17...|
|Perusahaan tidak ...|
|Rp1.000.000 - 2.0...|
|Rp7.000.000 - 12....|
| and designers to...|
|Rp13.000.000 - 18...|
|Rp2.000.000 - 3.0...|
|Rp5.000.000 - 6.0...|
|Rp1.000.000 - 1.5...|
|Perusahaan tidak ...|
+--------------------+



In [ ]:
format_salary_valid = spark.sql("""
SELECT
    salary_range,
    salary_range RLIKE '^Rp[0-9.]+ - [0-9.]+/Bulan$' AS cocok
FROM job_vacancy
WHERE salary_range LIKE 'Rp%'
LIMIT 20

""")

format_salary_valid.show(truncate=False)

+-------------------------------------------------------------------+-----+
|salary_range                                                       |cocok|
+-------------------------------------------------------------------+-----+
|Rp1.000.000 - 3.000.000/Bulan                                      |true |
|Rp6.500.000 - 7.500.000/Bulan                                      |true |
|Rp5.000.000 - 5.100.000/Bulan                                      |true |
|Rp1.000.000/Bulan                                                  |false|
|Rp5.729.876 - 6.000.000/Bulan                                      |true |
|Rp12.000.000 - 17.000.000/Bulan                                    |true |
|Rp1.000.000 - 2.000.000/Bulan                                      |true |
|Rp7.000.000 - 12.000.000/Bulan                                     |true |
|Rp13.000.000 - 18.000.000/Bulan                                    |true |
|Rp2.000.000 - 3.000.000/Bulan · Bonus Rp1.000.000 - 2.000.000/month|false|
|Rp5.000.000

In [ ]:
format_salary = spark.sql("""
SELECT
    CASE
        WHEN salary_range RLIKE '^Rp[0-9.]+ - [0-9.]+/Bulan$'
            THEN 'salary_range_valid'

        WHEN salary_range RLIKE '^Rp[0-9.]+/Bulan$'
            THEN 'single_salary'

        WHEN salary_range LIKE 'Perusahaan tidak%'
            THEN 'salary_hidden'

        ELSE 'noise'
    END AS kategori,
    COUNT(*) AS jumlah
FROM job_vacancy
GROUP BY 1

""")

format_salary.show(truncate=False)

+------------------+------+
|kategori          |jumlah|
+------------------+------+
|noise             |4253  |
|single_salary     |375   |
|salary_range_valid|10445 |
|salary_hidden     |1639  |
+------------------+------+



In [ ]:
df_kategori = spark.sql("""
SELECT
    salary_range,
    CASE
        WHEN salary_range RLIKE '^Rp[0-9.]+ - [0-9.]+/Bulan$'
            THEN 'salary_range_valid'

        WHEN salary_range RLIKE '^Rp[0-9.]+/Bulan$'
            THEN 'single_salary'

        WHEN salary_range LIKE 'Perusahaan tidak%'
            THEN 'salary_hidden'

        ELSE 'noise'
    END AS kategori
FROM job_vacancy
""")

for kategori in ["salary_range_valid", "single_salary", "salary_hidden", "noise"]:
    print(f"\n=== {kategori} ===")
    df_kategori.filter(df_kategori.kategori == kategori)\
               .select("salary_range")\
               .show(10, truncate=False)


=== salary_range_valid ===
+-------------------------------+
|salary_range                   |
+-------------------------------+
|Rp1.000.000 - 3.000.000/Bulan  |
|Rp6.500.000 - 7.500.000/Bulan  |
|Rp5.000.000 - 5.100.000/Bulan  |
|Rp5.729.876 - 6.000.000/Bulan  |
|Rp12.000.000 - 17.000.000/Bulan|
|Rp1.000.000 - 2.000.000/Bulan  |
|Rp7.000.000 - 12.000.000/Bulan |
|Rp13.000.000 - 18.000.000/Bulan|
|Rp5.000.000 - 6.000.000/Bulan  |
|Rp1.000.000 - 1.500.000/Bulan  |
+-------------------------------+
only showing top 10 rows

=== single_salary ===
+-----------------+
|salary_range     |
+-----------------+
|Rp1.000.000/Bulan|
|Rp3.000.000/Bulan|
|Rp4.000.000/Bulan|
|Rp8.000.000/Bulan|
|Rp1.000.000/Bulan|
|Rp4.000.000/Bulan|
|Rp5.000.000/Bulan|
|Rp1.111.111/Bulan|
|Rp700.000/Bulan  |
|Rp3.500.000/Bulan|
+-----------------+
only showing top 10 rows

=== salary_hidden ===
+---------------------------------+
|salary_range                     |
+---------------------------------+
|Perusahaan 

# **Data Cleaning**

In [ ]:
null_nan_checks = []
for c, dtype in df.dtypes:
    if dtype in ["int", "long", "float", "double", "short", "decimal"]:
        # For numeric columns, check for both NaN and null
        null_nan_checks.append(sum(when(isnan(col(c)) | col(c).isNull(), 1).otherwise(0)).alias(c))
    else:
        # For non-numeric columns (like string), only check for null
        null_nan_checks.append(sum(when(col(c).isNull(), 1).otherwise(0)).alias(c))

df.select(null_nan_checks).show()

+-----+-------+--------+--------+--------+----------------+---------------+------------+----------------+--------------------+-----------+----+---------------+
|title|company|location|job_type|category|experience_level|education_level|salary_range|job_requirements|job_responsibilities|posted_date| url|source_platform|
+-----+-------+--------+--------+--------+----------------+---------------+------------+----------------+--------------------+-----------+----+---------------+
|    0|   1497|    1683|    1939|    2300|            3055|           2497|        2547|            2579|                2631|       3716|3748|           3780|
+-----+-------+--------+--------+--------+----------------+---------------+------------+----------------+--------------------+-----------+----+---------------+



In [ ]:
df_clean = df.dropna()

In [ ]:
null_nan_checks = []
for c, dtype in df_clean.dtypes:
    if dtype in ["int", "long", "float", "double", "short", "decimal"]:
        # For numeric columns, check for both NaN and null
        null_nan_checks.append(sum(when(isnan(col(c)) | col(c).isNull(), 1).otherwise(0)).alias(c))
    else:
        # For non-numeric columns (like string), only check for null
        null_nan_checks.append(sum(when(col(c).isNull(), 1).otherwise(0)).alias(c))

df_clean.select(null_nan_checks).show()

+-----+-------+--------+--------+--------+----------------+---------------+------------+----------------+--------------------+-----------+---+---------------+
|title|company|location|job_type|category|experience_level|education_level|salary_range|job_requirements|job_responsibilities|posted_date|url|source_platform|
+-----+-------+--------+--------+--------+----------------+---------------+------------+----------------+--------------------+-----------+---+---------------+
|    0|      0|       0|       0|       0|               0|              0|           0|               0|                   0|          0|  0|              0|
+-----+-------+--------+--------+--------+----------------+---------------+------------+----------------+--------------------+-----------+---+---------------+



In [ ]:
print(f"Panjang baris: {df_clean.count()}")
print(f"Panjang kolom: {len(df_clean.columns)}")

Panjang baris: 12296
Panjang kolom: 13


In [ ]:
df.show(5)

+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+---------------+
|               title|             company|     location|            job_type|            category|    experience_level|     education_level|        salary_range|    job_requirements|job_responsibilities|posted_date|                 url|source_platform|
+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+---------------+
|Maintenance Engin...|        Manna Kampus|  Kab. Sleman|Penuh Waktu · Ker...|Komputer & Perang...|Pengalaman kurang...|Minimal Sarjana (S1)|Perusahaan tidak ...|Server Management...|🖥️ Tech Problem ...| 28-02-2026|https://glints.co...|  

In [ ]:
df_clean.show(5)

+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+---------------+
|               title|             company|     location|            job_type|            category|    experience_level|     education_level|        salary_range|    job_requirements|job_responsibilities|posted_date|                 url|source_platform|
+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+---------------+
|Maintenance Engin...|        Manna Kampus|  Kab. Sleman|Penuh Waktu · Ker...|Komputer & Perang...|Pengalaman kurang...|Minimal Sarjana (S1)|Perusahaan tidak ...|Server Management...|🖥️ Tech Problem ...| 28-02-2026|https://glints.co...|  

In [ ]:
total_rows = df_clean.count()
unique_rows = df_clean.distinct().count()

duplicates = total_rows - unique_rows

print("Duplicate rows:", duplicates)

Duplicate rows: 379


In [ ]:
df_clean = df_clean.dropDuplicates()


In [ ]:
total_rows = df_clean.count()
unique_rows = df_clean.distinct().count()

duplicates = total_rows - unique_rows

print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [ ]:
print(f"Panjang baris: {df_clean.count()}")
print(f"Panjang kolom: {len(df_clean.columns)}")

Panjang baris: 11917
Panjang kolom: 13


In [ ]:
df_kategori = df_clean.withColumn(
    "kategori_salary",
    when(
        col("salary_range").rlike(r"^Rp[0-9.]+ - [0-9.]+/Bulan$"),
        "salary_range_valid"
    ).when(
        col("salary_range").rlike(r"^Rp[0-9.]+/Bulan$"),
        "single_salary"
    ).when(
        col("salary_range").like("Perusahaan tidak%"),
        "salary_hidden"
    ).otherwise("noise")
)

In [ ]:
df_clean = df_kategori.filter(
    col("kategori_salary") != "noise"
)

In [ ]:
df_clean.show(5)

+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+---------------+------------------+
|               title|             company|       location|            job_type|            category|    experience_level|     education_level|        salary_range|    job_requirements|job_responsibilities|posted_date|                 url|source_platform|   kategori_salary|
+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+---------------+------------------+
|    Mobile Developer|PT Sakti Rekatama...|Jakarta Selatan|Penuh Waktu · Ker...|Komputer & Perang...|1 - 3 tahun penga...|Minimal Sarjana (S1)|Rp6.000.000 - 6.5...|Untuk melih

In [ ]:
print(f"Panjang baris: {df_clean.count()}")
print(f"Panjang kolom: {len(df_clean.columns)}")

Panjang baris: 10686
Panjang kolom: 14
